[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_USERNAME/YOUR_REPO/blob/main/Task2_RF_SVM_Implementation.ipynb)

# Algorithm Implementation and Code Analysis
### Random Forest and Support Vector Machine for Coffee Leaf Disease Classification

**Course:** Advanced Machine Learning (81203 ST) Course Work Two
**Programme:** MSc Computer Science
**Task:** Examine Python implementations of the selected model (Random Forest, SVM) using
appropriate libraries (Scikit-learn, OpenCV, scikit-image), explaining key functions,
parameters, and computational steps.

**Dataset:** RoCoLe *A Robusta Coffee Leaf Images Dataset* (Parraga-Alava, Cusme, Loor, &
Santander, 2019), 1,560 real-world Robusta coffee leaf images collected in Ecuador under
uncontrolled field conditions (natural lighting, cluttered backgrounds), labelled Healthy vs.
Diseased (leaf rust / red spider mite damage). Publicly available on
[Mendeley Data](https://doi.org/10.17632/c5yvn32dzg.2) and mirrored on
[Kaggle](https://www.kaggle.com/datasets/nirmalsankalana/rocole-a-robusta-coffee-leaf-images-dataset).
Uganda-specific field data was not yet available to the author at the time of writing (Task 1,
§2.3), so RoCoLe is used here as the closest available proxy Robusta dataset with agro-ecological
and photographic conditions comparable to Ugandan smallholder plantations.

**What this notebook does, step by step:**
1. Loads and organises the image dataset
2. Extracts hand-crafted features from each leaf image (colour, texture, shape) the
   feature-engineering step classical models such as RF/SVM require, since (unlike CNNs) they
   cannot learn features directly from raw pixels
3. Trains a **Random Forest** classifier and explains every constructor parameter
4. Trains a **Support Vector Machine** classifier and explains every constructor parameter
5. Evaluates and compares both models, including feature importance (RF) and support-vector
   analysis (SVM)

Every design choice below is cross-referenced to its formal derivation in **Task 6
(Mathematical Foundations)** of the coursework answer booklet.

> **How to run this in Google Colab:** Runtime → Change runtime type → CPU is sufficient (no
> GPU needed). Run the cells top to bottom. Cell 2 downloads the dataset directly from Kaggle —
> you will need a free Kaggle account and API token (`kaggle.json`), or you can upload the RoCoLe
> folder manually to `/content/RoCoLe` via the Colab file browser.

In [ ]:
# ---------------------------------------------------------------
# CELL 1: Environment setup and library imports
# ---------------------------------------------------------------
# numpy/pandas       -> numerical arrays and tabular feature storage
# opencv (cv2)       -> image loading, colour-space conversion, resizing
# scikit-image       -> texture (GLCM) and shape descriptors
# scikit-learn       -> RandomForestClassifier, SVC, preprocessing, metrics
# matplotlib/seaborn -> plots (confusion matrix, feature importance, ROC)

!pip -q install scikit-image kaggle

import os, glob, cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from skimage.feature import graycomatrix, graycoprops
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                              precision_recall_fscore_support, roc_auc_score, RocCurveDisplay)

import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)

print("Libraries imported successfully.")

In [ ]:
# ---------------------------------------------------------------
# CELL 2: Download the RoCoLe dataset
# ---------------------------------------------------------------
# Option A (recommended): Kaggle API. Upload your kaggle.json (Kaggle -> Account ->
# Create New API Token) using the file upload widget below, then run this cell.
from google.colab import files
import os

if not os.path.exists("/root/.kaggle/kaggle.json"):
    print("Upload your kaggle.json now (skip this if you already uploaded the dataset manually).")
    try:
        uploaded = files.upload()
        os.makedirs("/root/.kaggle", exist_ok=True)
        for fn in uploaded:
            os.rename(fn, "/root/.kaggle/kaggle.json")
        os.chmod("/root/.kaggle/kaggle.json", 0o600)
        !kaggle datasets download -d nirmalsankalana/rocole-a-robusta-coffee-leaf-images-dataset -p /content --unzip
    except Exception as e:
        print("Kaggle upload skipped/failed:", e)
        print("Manually upload the RoCoLe image folders to /content/RoCoLe instead.")

DATA_DIR = "/content"  # adjust if your unzipped folder has a different root
print("Dataset root ready at:", DATA_DIR)

In [ ]:
# ---------------------------------------------------------------
# CELL 3: Build a (filepath, label) index
# ---------------------------------------------------------------
# RoCoLe ships as class-labelled subfolders (e.g. Healthy/, Rust/, Red_spider_mite/).
# We collapse the problem to the binary framing used in Task 1/3: Healthy vs Diseased,
# by treating every non-Healthy folder as the "Diseased" class.

def index_dataset(root, healthy_keyword="healthy"):
    records = []
    for path in Path(root).rglob("*"):
        if path.suffix.lower() in (".jpg", ".jpeg", ".png"):
            label = "Healthy" if healthy_keyword in path.parent.name.lower() else "Diseased"
            records.append({"filepath": str(path), "label": label})
    return pd.DataFrame(records)

df = index_dataset(DATA_DIR)
print(df["label"].value_counts())
df.head()

## Feature Engineering

Random Forest and SVM, unlike CNNs, do not learn image representations end-to-end they need
**hand-crafted numerical features** as input. Three complementary feature families are extracted,
matching the feature types most consistently cited across the reviewed coffee-disease literature
(colour, texture, shape see Task 1, §3.1 "Handles Mixed Data Types"):

- **Colour histogram (HSV space, 3×16 bins = 48 features):** disease lesions (rust orange-brown,
  cercospora grey) shift the hue/saturation distribution away from healthy green.
- **Gray-Level Co-occurrence Matrix, GLCM (contrast, homogeneity, energy, correlation = 4
  features):** captures leaf-surface texture disruption caused by lesions and necrosis.
- **Hu moments (7 features):** rotation/scale/translation-invariant shape descriptors capturing
  lesion-driven changes in leaf silhouette/blob shape.

Total feature vector length: 48 + 4 + 7 = **59 features per image**.

In [ ]:
# ---------------------------------------------------------------
# CELL 4: Feature extraction functions
# ---------------------------------------------------------------
IMG_SIZE = 224  # standard resize so every feature vector has a comparable scale

def color_histogram_features(img_bgr, bins=16):
    """HSV colour histogram: captures disease-driven colour shifts (green -> brown/orange/grey)."""
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    hist = []
    for ch in range(3):
        h = cv2.calcHist([hsv], [ch], None, [bins], [0, 256])
        h = cv2.normalize(h, h).flatten()   # normalise so histogram is scale-invariant to leaf size
        hist.extend(h)
    return np.array(hist)

def texture_features(img_bgr):
    """GLCM texture descriptors: quantify surface roughness/uniformity disrupted by lesions."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    glcm = graycomatrix(gray, distances=[1], angles=[0], levels=256, symmetric=True, normed=True)
    return np.array([
        graycoprops(glcm, "contrast")[0, 0],
        graycoprops(glcm, "homogeneity")[0, 0],
        graycoprops(glcm, "energy")[0, 0],
        graycoprops(glcm, "correlation")[0, 0],
    ])

def shape_features(img_bgr):
    """Hu moments: 7 invariant moments describing the leaf/lesion blob shape."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    moments = cv2.moments(thresh)
    hu = cv2.HuMoments(moments).flatten()
    return -np.sign(hu) * np.log10(np.abs(hu) + 1e-10)  # log-scale: Hu moments span many orders of magnitude

def extract_features(filepath):
    img = cv2.imread(filepath)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    return np.concatenate([
        color_histogram_features(img),
        texture_features(img),
        shape_features(img),
    ])

print("Feature vector length:", len(extract_features(df.iloc[0]["filepath"])))

In [ ]:
# ---------------------------------------------------------------
# CELL 5: Extract features for the full dataset
# ---------------------------------------------------------------
X_list, y_list = [], []
for _, row in df.iterrows():
    try:
        X_list.append(extract_features(row["filepath"]))
        y_list.append(row["label"])
    except Exception as e:
        print("Skipped", row["filepath"], "->", e)

X = np.array(X_list)
le = LabelEncoder()
y = le.fit_transform(y_list)  # Healthy=0, Diseased=1 (alphabetical)

print("X shape:", X.shape, " y shape:", y.shape)
print("Classes:", dict(zip(le.classes_, range(len(le.classes_)))))

## Train / Test Split and Scaling

- `train_test_split(..., stratify=y)` preserves the Healthy:Diseased class ratio in both splits
  important because RoCoLe (like most field datasets) is imbalanced.
- `StandardScaler` is fitted **only on the training set** then applied to both splits, to avoid
  data leakage. Scaling matters for **SVM** (distance/margin-based, sensitive to feature scale;
  see Task 6, §2.1) but is not required for **Random Forest** (threshold-based splits are
  scale-invariant; Task 6, §1.2) we keep both a scaled and unscaled copy to demonstrate this.

In [ ]:
# ---------------------------------------------------------------
# CELL 6: Split and scale
# ---------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit on train only
X_test_scaled = scaler.transform(X_test)          # reuse train statistics on test

print("Train:", X_train.shape, " Test:", X_test.shape)

## Random Forest Implementation and Parameter Analysis

`RandomForestClassifier` builds `n_estimators` decision trees, each trained on a **bootstrap
sample** of the training data (bagging) and, at every split, considering only a random subset of
`max_features`this de-correlates the trees so their errors partially cancel out when the
ensemble votes (majority vote for classification). The variance-reduction argument for why this
works is derived formally in **Task 6, §1.1**.

Key parameters explained:

| Parameter | Meaning | Value used | Why |
|---|---|---|---|
| `n_estimators` | number of trees in the forest | 300 | more trees stabilise the ensemble vote; returns diminish after a few hundred |
| `criterion` | split-quality measure | `"gini"` | Gini impurity (Task 6, §1.2) is cheaper to compute than entropy and gives near-identical splits in practice |
| `max_depth` | maximum tree depth | `None` (grown until pure/`min_samples_leaf`) | RF controls overfitting via averaging across trees, not depth limits |
| `max_features` | features considered per split | `"sqrt"` | √59 ≈ 8; standard default for classification; introduces the randomness that decorrelates trees |
| `bootstrap` | sample with replacement per tree | `True` | enables bagging; also enables out-of-bag (OOB) error estimation (Task 6, §1.4) |
| `class_weight` | reweight loss for imbalanced classes | `"balanced"` | RoCoLe's Healthy/Diseased split is not perfectly 50/50 |
| `random_state` | reproducibility seed | 42 | deterministic results across reruns |

In [ ]:
# ---------------------------------------------------------------
# CELL 7: Random Forest training
# ---------------------------------------------------------------
rf = RandomForestClassifier(
    n_estimators=300,
    criterion="gini",
    max_depth=None,
    max_features="sqrt",
    bootstrap=True,
    oob_score=True,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42,
)
rf.fit(X_train, y_train)  # note: unscaled features are fine for RF

rf_pred = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:, 1]

print("Random Forest test accuracy:", accuracy_score(y_test, rf_pred))
print("Random Forest OOB score (Task 6, \u00a71.4):", rf.oob_score_)
print(classification_report(y_test, rf_pred, target_names=le.classes_))

In [ ]:
# ---------------------------------------------------------------
# CELL 8: Random Forest - feature importance (interpretability)
# ---------------------------------------------------------------
feature_names = (
    [f"H_bin{i}" for i in range(16)] + [f"S_bin{i}" for i in range(16)] + [f"V_bin{i}" for i in range(16)]
    + ["contrast", "homogeneity", "energy", "correlation"]
    + [f"hu{i+1}" for i in range(7)]
)
importances = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=False)

plt.figure(figsize=(8, 6))
importances.head(15).plot(kind="barh")
plt.gca().invert_yaxis()
plt.title("Random Forest - Top 15 Feature Importances (Mean Gini Decrease)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

print(importances.head(10))

**Interpretation:** feature importance in RF is computed as the mean decrease in Gini
impurity that each feature contributes across all trees/splits, normalised to sum to 1. Features
that consistently produce cleaner splits (larger class separation) score higher. In coffee leaf
disease classification, colour-histogram bins (hue/saturation) typically dominate, matching the
Task 1 justification that RF's transparent, ranked feature importance is valuable for farmer/
extension-worker trust unlike a CNN's opaque convolutional filters.

## Support Vector Machine Implementation and Parameter Analysis

`SVC` finds the hyperplane that maximises the margin between classes (Task 6, §2.1–2.2). With the
RBF kernel, the implicit feature mapping (Task 6, §2.4) lets SVM separate classes that are not
linearly separable in the original 59-dimensional feature space.

Key parameters explained:

| Parameter | Meaning | Value used | Why |
|---|---|---|---|
| `kernel` | similarity function used to compare samples | `"rbf"` | disease colour/texture boundaries are non-linear; RBF is the standard default for image-derived features |
| `C` | regularisation — trade-off between margin width and misclassification | grid-searched over `[0.1, 1, 10, 100]` | small `C` = wider margin/more tolerant of errors; large `C` = fits training data tightly (risk of overfitting) formalised in Task 6, §2.2 |
| `gamma` | RBF kernel width (`1/(2σ²)`), controls how far a single sample's influence reaches | grid-searched over `["scale", 0.01, 0.001]` | large `gamma` = very local decision boundary (overfit risk); small `gamma` = smoother boundary |
| `class_weight` | reweight loss for imbalanced classes | `"balanced"` | mirrors the RF setting for a fair comparison |
| `probability` | enable `predict_proba` via Platt scaling | `True` | needed for ROC-AUC and confidence scores (Task 1's "probability calibration" strength) |

Note SVM is trained on `X_train_scaled`, **not** `X_train`SVM's margin computation is
distance-based, so unscaled features (colour bins 0-1 vs. GLCM contrast in the hundreds) would let
high-magnitude features dominate the decision boundary purely due to units, not information
content.

In [ ]:
# ---------------------------------------------------------------
# CELL 9: SVM training with grid search over C and gamma
# ---------------------------------------------------------------
param_grid = {
    "C": [0.1, 1, 10, 100],
    "gamma": ["scale", 0.01, 0.001],
    "kernel": ["rbf"],
}

svm_base = SVC(class_weight="balanced", probability=True, random_state=42)
grid = GridSearchCV(svm_base, param_grid, cv=StratifiedKFold(5), scoring="f1", n_jobs=-1)
grid.fit(X_train_scaled, y_train)

print("Best hyperparameters:", grid.best_params_)
svm = grid.best_estimator_

svm_pred = svm.predict(X_test_scaled)
svm_proba = svm.predict_proba(X_test_scaled)[:, 1]

print("SVM test accuracy:", accuracy_score(y_test, svm_pred))
print(classification_report(y_test, svm_pred, target_names=le.classes_))

In [ ]:
# ---------------------------------------------------------------
# CELL 10: SVM - support vector analysis
# ---------------------------------------------------------------
n_sv = svm.n_support_
print(f"Support vectors per class: Healthy={n_sv[0]}, Diseased={n_sv[1]}")
print(f"Total support vectors: {sum(n_sv)} out of {len(X_train_scaled)} training samples "
      f"({100*sum(n_sv)/len(X_train_scaled):.1f}%)")
print("\nThis confirms Task 1's memory-efficiency claim (and the KKT/support-vector argument in "
      "Task 6, \u00a72.3): only the samples closest to the decision boundary are retained; the rest "
      "of the training set can be discarded at inference time.")

## Model Comparison and Evaluation

In [ ]:
# ---------------------------------------------------------------
# CELL 11: Side-by-side evaluation - confusion matrices and ROC curves
# ---------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (name, pred) in zip(axes, [("Random Forest", rf_pred), ("SVM", svm_pred)]):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=le.classes_,
                yticklabels=le.classes_, ax=ax)
    ax.set_title(f"{name} - Confusion Matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_predictions(y_test, rf_proba, name="Random Forest", ax=ax)
RocCurveDisplay.from_predictions(y_test, svm_proba, name="SVM (RBF)", ax=ax)
ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
ax.set_title("ROC Curves - Random Forest vs SVM")
plt.tight_layout()
plt.show()

summary = pd.DataFrame({
    "Model": ["Random Forest", "SVM (RBF)"],
    "Accuracy": [accuracy_score(y_test, rf_pred), accuracy_score(y_test, svm_pred)],
    "F1 (Diseased)": [
        precision_recall_fscore_support(y_test, rf_pred, average="binary")[2],
        precision_recall_fscore_support(y_test, svm_pred, average="binary")[2],
    ],
    "ROC-AUC": [roc_auc_score(y_test, rf_proba), roc_auc_score(y_test, svm_proba)],
})
summary

In [ ]:
# ---------------------------------------------------------------
# CELL 12: 5-fold cross-validation (robustness check beyond a single split)
# ---------------------------------------------------------------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_cv = cross_val_score(rf, X, y, cv=cv, scoring="accuracy")
svm_cv = cross_val_score(svm, scaler.fit_transform(X), y, cv=cv, scoring="accuracy")

print(f"Random Forest 5-fold CV accuracy: {rf_cv.mean():.4f} (+/- {rf_cv.std():.4f})")
print(f"SVM 5-fold CV accuracy:           {svm_cv.mean():.4f} (+/- {svm_cv.std():.4f})")

## Summary Computational Steps Recap

1. **Feature extraction** (O(n) over images): colour histogram, GLCM texture, Hu-moment shape
   converts each image into a fixed-length numeric vector so classical ML models can operate on it.
2. **Random Forest fit**: for each of 300 trees, bootstrap-sample the training rows, grow a tree by
   recursively selecting the best Gini split from a random √59 ≈ 8-feature subset at each node,
   until leaves are pure or `min_samples_leaf` is reached. Prediction = majority vote across trees
   (Task 6, §1.3).
3. **SVM fit**: solve the constrained quadratic program (dual form, Task 6, §2.3) to find the
   support vectors and Lagrange multipliers that define the maximum-margin RBF-kernel boundary,
   via `libsvm`'s SMO (Sequential Minimal Optimization) solver under the hood.
4. **Evaluation**: accuracy, F1, ROC-AUC on a held-out stratified test split, plus 5-fold
   cross-validation for a variance-aware performance estimate.

*Paste the shareable Colab link for this notebook into Task 2 of the coursework answer booklet.*